<h1><center>Agentic AI: Responder and Verifier </center></h1>

In [ ]:
import os
import json
import glob
import shutil
import hashlib
from dataclasses import dataclass, field
from typing import Any, Dict, List
from importlib.metadata import version
from pydantic import BaseModel, Field
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.tools import create_retriever_tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
import sys
import atexit
import re
import langchain
import langchain_core
import builtins


# os.environ["OPENAI_API_KEY"] = "xxxxxxxxxxxxx"
with open("OPENAI_API_KEY.txt", "r", encoding="utf-8") as f:
    os.environ["OPENAI_API_KEY"] = f.read().strip()

OUTPUT_DIR = "outputs"
COT_STEP_RESULTS_PATH = os.path.join(OUTPUT_DIR, "cot_step_results.json")
COT_FINAL_DRAFT_PATH = os.path.join(OUTPUT_DIR, "cot_final_draft.txt")
LOG_PATH = os.path.join(OUTPUT_DIR, "log.txt")

# === Model selection ===
PHASE1_MODEL_NAME = "gpt-5.4-2026-03-05"
RESPONDER_MODEL_NAME = "gpt-5.4-2026-03-05"
VERIFIER_MODEL_NAME = "gpt-5.4-2026-03-05"
DEFAULT_TEMPERATURE = 0

def ensure_output_dir(output_dir: str) -> None:
    os.makedirs(output_dir, exist_ok=True)


def save_text_file(path: str, content: str) -> None:
    ensure_output_dir(os.path.dirname(path) or ".")
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)


def save_json_file(path: str, data: Any) -> None:
    ensure_output_dir(os.path.dirname(path) or ".")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def initialize_log_file(log_path: str) -> None:
    ensure_output_dir(os.path.dirname(log_path) or ".")
    with open(log_path, "w", encoding="utf-8"):
        pass


def module_print(*args, **kwargs):
    builtins.print(*args, **kwargs)

    sep = kwargs.get("sep", " ")
    end = kwargs.get("end", "\n")
    text = sep.join(str(arg) for arg in args) + end

    ensure_output_dir(os.path.dirname(LOG_PATH) or ".")
    with open(LOG_PATH, "a", encoding="utf-8") as log_file:
        log_file.write(text)

initialize_log_file(LOG_PATH)
print = module_print        


## Check LangChain version
print("langchain:", langchain.__version__)
print("langchain_core:", langchain_core.__version__)
print(f"Log file will be saved to: {LOG_PATH}")


#######################################
# Build or load the RAG database
#######################################

persist_directory = "db20"
collection_name = "langchain"
source_directory = "./database"
meta_path = os.path.join(persist_directory, "index_meta.json")

splitter_config = {
    "separators": [
        "\n\\subsection{",
        "\n\\paragraph{",
        "\n\\begin{equation}",
        "\n\\begin{align}",
        "\n\n",
        "\n",
        " "
    ],
    "chunk_size": 1800,
    "chunk_overlap": 300,
    "length_function": len,
    "is_separator_regex": False,
}

embedding_model_name = "text-embedding-ada-002"
embedding = OpenAIEmbeddings(model=embedding_model_name)


def file_sha256(filepath: str) -> str:
    sha = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha.update(chunk)
    return sha.hexdigest()


def build_index_signature(source_dir: str, splitter_cfg: dict, embedding_model: str, collection: str):
    txt_files = sorted(glob.glob(os.path.join(source_dir, "*.txt")))
    file_records = []

    for path in txt_files:
        file_records.append({
            "path": os.path.abspath(path),
            "sha256": file_sha256(path),
        })

    payload = {
        "source_directory": os.path.abspath(source_dir),
        "files": file_records,
        "splitter_config": {
            "separators": splitter_cfg["separators"],
            "chunk_size": splitter_cfg["chunk_size"],
            "chunk_overlap": splitter_cfg["chunk_overlap"],
            "length_function": "len",
            "is_separator_regex": splitter_cfg["is_separator_regex"],
        },
        "embedding_model": embedding_model,
        "collection_name": collection,
        "vector_space": "cosine",
    }

    payload_json = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    signature = hashlib.sha256(payload_json.encode("utf-8")).hexdigest()
    return payload, signature


def should_rebuild_db(persist_dir: str, meta_file: str, current_signature: str) -> tuple[bool, str]:
    if not os.path.isdir(persist_dir):
        return True, f"'{persist_dir}' does not exist. Building a new RAG database."

    if not os.path.isfile(meta_file):
        return True, f"'{meta_file}' does not exist. Rebuilding the RAG database."

    try:
        with open(meta_file, "r", encoding="utf-8") as f:
            saved_meta = json.load(f)
    except Exception as e:
        return True, f"Failed to read index metadata: {e}. Rebuilding the RAG database."

    saved_signature = saved_meta.get("signature")
    if saved_signature != current_signature:
        return True, "Source files or indexing settings changed. Rebuilding the RAG database."

    try:
        test_db = Chroma(
            persist_directory=persist_dir,
            embedding_function=embedding,
            collection_name=collection_name,
        )
        doc_count = test_db._collection.count()
        if doc_count <= 0:
            return True, "Existing Chroma collection is empty. Rebuilding the RAG database."
    except Exception as e:
        return True, f"Failed to load existing Chroma database: {e}. Rebuilding the RAG database."

    return False, f"Found an existing valid RAG database in '{persist_dir}'. Loading it directly."


current_meta_payload, current_signature = build_index_signature(
    source_dir=source_directory,
    splitter_cfg=splitter_config,
    embedding_model=embedding_model_name,
    collection=collection_name,
)

rebuild_db, rebuild_reason = should_rebuild_db(
    persist_dir=persist_directory,
    meta_file=meta_path,
    current_signature=current_signature,
)
print(rebuild_reason)

if rebuild_db:
    txt_files = sorted(glob.glob(os.path.join(source_directory, "*.txt")))
    if not txt_files:
        raise ValueError(f"No .txt files were found in '{source_directory}'. Cannot build the RAG database.")

    if os.path.isdir(persist_directory):
        shutil.rmtree(persist_directory)

    loader = DirectoryLoader(
        source_directory,
        glob="*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    documents = loader.load()

    if not documents:
        raise ValueError(f"No documents were loaded from '{source_directory}'.")

    text_splitter = RecursiveCharacterTextSplitter(
        separators=splitter_config["separators"],
        chunk_size=splitter_config["chunk_size"],
        chunk_overlap=splitter_config["chunk_overlap"],
        length_function=splitter_config["length_function"],
        is_separator_regex=splitter_config["is_separator_regex"],
    )
    texts = text_splitter.split_documents(documents)

    if not texts:
        raise ValueError("Text splitting returned 0 chunks. Cannot build the RAG database.")

    vectordb = Chroma.from_documents(
        documents=texts,
        embedding=embedding,
        persist_directory=persist_directory,
        collection_name=collection_name,
        collection_configuration={"hnsw": {"space": "cosine"}},
    )

    os.makedirs(persist_directory, exist_ok=True)
    current_meta_payload["signature"] = current_signature
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(current_meta_payload, f, ensure_ascii=False, indent=2)

    print(f"RAG database built successfully. Number of chunks indexed: {len(texts)}")

else:
    vectordb = Chroma(
        persist_directory=persist_directory,
        embedding_function=embedding,
        collection_name=collection_name,
    )

    print(f"Loaded existing RAG database successfully. Number of stored chunks: {vectordb._collection.count()}")


chroma_version = version("langchain-chroma")
print(f"Chroma version: {chroma_version}")

retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print(f"retriever.search_type is: {retriever.search_type}")
print(f"retriever.search_kwargs is: {retriever.search_kwargs}")

#######################################
# Two-Agent CoT Pipeline
#######################################

Switch_Debug = True    
DEBUG_LOG_DIR = os.path.join(OUTPUT_DIR, "debug_logs")
MAX_RETRIES = 6

USER_PROMPT_PATH = "User_Prompt.txt"
SYSTEM_PROMPT_1_PATH = "System_Prompt1.txt"
SYSTEM_PROMPT_2_PATH = "System_Prompt2.txt"
SYSTEM_PROMPT_3_PATH = "System_Prompt3.txt"
GROUND_TRUTH_DIR = "ground_truth"

def load_text_file(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

system_prompt_1 = load_text_file(SYSTEM_PROMPT_1_PATH)
system_prompt_2 = load_text_file(SYSTEM_PROMPT_2_PATH)
system_prompt_3 = load_text_file(SYSTEM_PROMPT_3_PATH)
user_prompt_full = load_text_file(USER_PROMPT_PATH)

print(f"Loaded System_Prompt1 ({len(system_prompt_1)} chars)")
print(f"Loaded System_Prompt2 ({len(system_prompt_2)} chars)")
print(f"Loaded System_Prompt3 ({len(system_prompt_3)} chars)")
print(f"Loaded User_Prompt    ({len(user_prompt_full)} chars)")


def split_user_prompt(text: str) -> tuple[str, str]:
    marker = "[Chain-of-Thought Planning]"
    idx = text.find(marker)
    if idx == -1:
        raise ValueError("Cannot find '[Chain-of-Thought Planning]' section in User_Prompt.txt")
    part_a = text[:idx].strip()
    part_b = text[idx:].strip()
    return part_a, part_b

user_prompt_part_a, user_prompt_part_b = split_user_prompt(user_prompt_full)
print(f"User prompt split: Part-A length={len(user_prompt_part_a)}, Part-B length={len(user_prompt_part_b)}")

tool = create_retriever_tool(
    retriever,
    "search_by_similarity",
    "Searches and returns the most relevant content that can answer the question from the local document database."
)
tools = [tool]

planning_llm = ChatOpenAI(
    model=PHASE1_MODEL_NAME,
    temperature=DEFAULT_TEMPERATURE
)

responder_llm = ChatOpenAI(
    model=RESPONDER_MODEL_NAME,
    temperature=DEFAULT_TEMPERATURE
)

verifier_llm = ChatOpenAI(
    model=VERIFIER_MODEL_NAME,
    temperature=DEFAULT_TEMPERATURE
)

print(f"Phase 1 model: {PHASE1_MODEL_NAME}")
print(f"Responder model: {RESPONDER_MODEL_NAME}")
print(f"Verifier model: {VERIFIER_MODEL_NAME}")
print(f"Default temperature: {DEFAULT_TEMPERATURE}")

responder_memory = InMemorySaver()
responder_agent = create_agent(
    model=responder_llm,
    tools=tools,
    system_prompt=system_prompt_2,
    checkpointer=responder_memory,
)
responder_thread = {"configurable": {"thread_id": "responder_solving"}}
print("Responder Agent created (Phase-2 Solving, with RAG tool + memory)")

verifier_memory = InMemorySaver()
verifier_agent = create_agent(
    model=verifier_llm,
    tools=tools,
    system_prompt=system_prompt_3,
    checkpointer=verifier_memory,
)
verifier_thread = {"configurable": {"thread_id": "verifier"}}
print("Verifier Agent created (with RAG tool + System_Prompt3 + memory)")


# === Helper Functions ===

def invoke_agent(agent, message_content: str, thread_config: dict) -> tuple[str, list]:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": message_content}]},
        thread_config,
    )
    response_text = result["messages"][-1].content
    return response_text, result["messages"]


def format_messages_for_debug(messages: list) -> str:
    lines = []
    for i, msg in enumerate(messages):
        msg_type = getattr(msg, "type", "unknown")
        content = getattr(msg, "content", "")
        if isinstance(content, list):
            content = json.dumps(content, ensure_ascii=False, indent=2)
        elif not isinstance(content, str):
            content = str(content)
        lines.append(f"--- Message {i} [{msg_type}] ---\n{content}")
    return "\n\n".join(lines)


def is_verified(verifier_response: str) -> bool:
    return verifier_response.strip().startswith("Your step response is correct")


def build_final_draft_from_step_results(cot_step_results: dict) -> str:
    sections = []

    for step in cot_step_results.get("phase2_solving", []):
        step_number = step.get("step_number")
        attempts = step.get("attempts", [])

        if not attempts:
            continue

        last_answer = attempts[-1].get("responder_output", "").strip()
        if not last_answer:
            continue

        sections.append(
            f"% ===== CoT Step {step_number} =====\n"
            f"{last_answer}"
        )

    return "\n\n".join(sections).strip()


# === cot_step_results tracking ===
cot_step_results = {
    "phase1_planning": {},
    "phase2_solving": []
}


###############################################################################
# PHASE 1: Planning — Parse CoT steps from User_Prompt
###############################################################################
print("\n" + "=" * 60)
print("PHASE 1: Planning — CoT Step Parsing")
print("=" * 60)

phase1_user_message = (
    f"{user_prompt_full}\n\n"
    "---\n"
    "IMPORTANT: Output your result as a JSON array ONLY, with no additional text, "
    "no markdown fences, and no preamble. Each element must have the following keys:\n"
    '  "step_number": integer,\n'
    '  "step_title": string,\n'
    '  "step_query": string'
)

print("Sending planning request to LLM (direct call, no agent)...")
phase1_response = planning_llm.invoke([
    SystemMessage(content=system_prompt_1),
    HumanMessage(content=phase1_user_message),
])
phase1_raw = phase1_response.content
print(f"Phase 1 raw response length: {len(phase1_raw)} chars")

phase1_clean = re.sub(r"^```(?:json)?\s*", "", phase1_raw.strip())
phase1_clean = re.sub(r"\s*```$", "", phase1_clean.strip())

try:
    cot_steps = json.loads(phase1_clean)
except json.JSONDecodeError as e:
    print(f"ERROR: Failed to parse Phase 1 JSON output: {e}")
    print(f"Raw output:\n{phase1_raw}")
    raise

num_steps = len(cot_steps)
print(f"Identified {num_steps} CoT step(s)")

for step in cot_steps:
    step_num = step["step_number"]
    step_path = os.path.join(OUTPUT_DIR, f"CoT_step_query{step_num}.txt")
    save_text_file(step_path, step["step_query"])
    print(f"  Saved: {step_path}")

cot_step_results["phase1_planning"] = {
    "input_system_prompt_file": SYSTEM_PROMPT_1_PATH,
    "input_user_message": phase1_user_message,
    "raw_output": phase1_raw,
    "parsed_steps": cot_steps,
    "num_steps": num_steps,
}

if Switch_Debug:
    ensure_output_dir(DEBUG_LOG_DIR)
    debug_content = (
        f"=== PHASE 1: Planning ===\n\n"
        f"--- System Prompt (from {SYSTEM_PROMPT_1_PATH}) ---\n{system_prompt_1}\n\n"
        f"--- User Message ---\n{phase1_user_message}\n\n"
        f"--- LLM Response ---\n{phase1_raw}\n"
    )
    save_text_file(os.path.join(DEBUG_LOG_DIR, "phase1_planning_full.txt"), debug_content)
    print(f"  Debug log saved: {os.path.join(DEBUG_LOG_DIR, 'phase1_planning_full.txt')}")


###############################################################################
# PHASE 2: Solving — Iterative Responder / Verifier Loop
###############################################################################
print("\n" + "=" * 60)
print("PHASE 2: Solving — Iterative Responder / Verifier Loop")
print("=" * 60)

for step_idx in range(1, num_steps + 1):
    step_query_path = os.path.join(OUTPUT_DIR, f"CoT_step_query{step_idx}.txt")
    step_query = load_text_file(step_query_path)

    gt_path = os.path.join(GROUND_TRUTH_DIR, f"CoT_step_query{step_idx}_ground_truth.txt")
    if os.path.isfile(gt_path):
        ground_truth = load_text_file(gt_path)
    else:
        print(f"  WARNING: Ground truth not found: {gt_path}. Skipping verification for step {step_idx}.")
        ground_truth = None

    step_result = {
        "step_number": step_idx,
        "step_query_file": f"CoT_step_query{step_idx}.txt",
        "step_query": step_query,
        "attempts": [],
        "final_status": None,
    }

    verified = False
    verifier_feedback = ""

    for attempt in range(1, MAX_RETRIES + 1):
        print(f"\n--- Step {step_idx}, Attempt {attempt} ---")

        attempt_record = {
            "attempt": attempt,
            "responder_input": None,
            "responder_output": None,
            "verifier_input": None,
            "verifier_output": None,
            "verified": False,
        }

        if attempt == 1:
            if step_idx == 1:
                responder_msg = (
                    f"Here is the system description and required mathematical formulation "
                    f"for your reference:\n\n"
                    f"{user_prompt_part_a}\n\n"
                    f"---\n\n"
                    f"Now please answer the following step query:\n\n"
                    f"{step_query}"
                )
            else:
                responder_msg = (
                    f"Please answer the following step query:\n\n"
                    f"{step_query}"
                )
        else:
            responder_msg = verifier_feedback

        attempt_record["responder_input"] = responder_msg

        print(f"  Sending to Responder Agent...")
        responder_answer, responder_messages = invoke_agent(
            responder_agent, responder_msg, responder_thread
        )
        attempt_record["responder_output"] = responder_answer

        answer_path = os.path.join(OUTPUT_DIR, f"CoT_step_query{step_idx}_answer{attempt}.txt")
        save_text_file(answer_path, responder_answer)
        print(f"  Responder answer saved: {answer_path}")
        print(f"  Responder answer length: {len(responder_answer)} chars")

        if Switch_Debug:
            ensure_output_dir(DEBUG_LOG_DIR)
            debug_filename = f"CoT_step_query{step_idx}_full_R{attempt}.txt"
            debug_content = (
                f"=== Responder Agent: Step {step_idx}, Attempt {attempt} ===\n\n"
                f"--- Agent System Prompt (from {SYSTEM_PROMPT_2_PATH}) ---\n"
                f"{system_prompt_2}\n\n"
                f"--- User Message Sent This Turn ---\n{responder_msg}\n\n"
                f"--- Full Message History (including short-term memory) ---\n"
                f"{format_messages_for_debug(responder_messages)}\n"
            )
            save_text_file(os.path.join(DEBUG_LOG_DIR, debug_filename), debug_content)
            print(f"  Debug log saved: {os.path.join(DEBUG_LOG_DIR, debug_filename)}")

        if ground_truth is None:
            print(f"  No ground truth available. Marking step {step_idx} as verified.")
            verified = True
            attempt_record["verified"] = True
            step_result["attempts"].append(attempt_record)
            break

        verifier_msg = (
            f"Please compare the following two documents for semantic consistency. "
            f"Where applicable, verify that mathematical formulas, symbols, and their "
            f"explanations are semantically equivalent.\n\n"
            f"=== Responder Answer ===\n{responder_answer}\n\n"
            f"=== Ground Truth ===\n{ground_truth}\n\n"
            f"The original step query was:\n{step_query}\n\n"
            f"Provide your verification feedback as instructed in your system prompt."
        )
        attempt_record["verifier_input"] = verifier_msg

        print(f"  Sending to Verifier Agent...")
        verifier_feedback, verifier_messages = invoke_agent(
            verifier_agent, verifier_msg, verifier_thread
        )
        attempt_record["verifier_output"] = verifier_feedback

        verif_path = os.path.join(OUTPUT_DIR, f"CoT_step_query{step_idx}_verif{attempt}.txt")
        save_text_file(verif_path, verifier_feedback)
        print(f"  Verifier output saved: {verif_path}")

        if Switch_Debug:
            debug_filename_v = f"CoT_step_query{step_idx}_verif_full_R{attempt}.txt"
            debug_content_v = (
                f"=== Verifier Agent: Step {step_idx}, Attempt {attempt} ===\n\n"
                f"--- Agent System Prompt (from {SYSTEM_PROMPT_3_PATH}) ---\n"
                f"{system_prompt_3}\n\n"
                f"--- User Message Sent This Turn ---\n{verifier_msg}\n\n"
                f"--- Full Message History ---\n"
                f"{format_messages_for_debug(verifier_messages)}\n"
            )
            save_text_file(os.path.join(DEBUG_LOG_DIR, debug_filename_v), debug_content_v)

        if is_verified(verifier_feedback):
            print(f"  VERIFIED: Step {step_idx} passed on attempt {attempt}.")
            verified = True
            attempt_record["verified"] = True
            step_result["attempts"].append(attempt_record)
            break
        else:
            print(f"  REJECTED: Step {step_idx} failed on attempt {attempt}.")
            attempt_record["verified"] = False
            step_result["attempts"].append(attempt_record)

    step_result["final_status"] = "verified" if verified else "max_retries_reached"
    if not verified:
        print(f"  WARNING: Step {step_idx} reached max retries ({MAX_RETRIES}) without verification.")

    cot_step_results["phase2_solving"].append(step_result)
    save_json_file(COT_STEP_RESULTS_PATH, cot_step_results)

save_json_file(COT_STEP_RESULTS_PATH, cot_step_results)

final_draft = build_final_draft_from_step_results(cot_step_results)
save_text_file(COT_FINAL_DRAFT_PATH, final_draft)

print(f"\nCoT step results saved to: {COT_STEP_RESULTS_PATH}")
print(f"Final integrated draft saved to: {COT_FINAL_DRAFT_PATH}")

print("\n" + "=" * 60)
print("FINAL INTEGRATED DRAFT")
print("=" * 60)
print(final_draft if final_draft else "[EMPTY FINAL DRAFT]")

print("\n" + "=" * 60)
print("Pipeline completed.")
print("=" * 60)

langchain: 1.2.13
langchain_core: 1.2.21
Log file will be saved to: outputs\log.txt
Found an existing valid RAG database in 'db20'. Loading it directly.
Loaded existing RAG database successfully. Number of stored chunks: 19
Chroma version: 1.1.0
retriever.search_type is: similarity
retriever.search_kwargs is: {'k': 4}
Loaded System_Prompt1 (4272 chars)
Loaded System_Prompt2 (4155 chars)
Loaded System_Prompt3 (2933 chars)
Loaded User_Prompt    (5558 chars)
User prompt split: Part-A length=4104, Part-B length=1452
Phase 1 model: gpt-5.4-2026-03-05
Responder model: gpt-5.4-2026-03-05
Verifier model: gpt-5.4-2026-03-05
Default temperature: 0
Responder Agent created (Phase-2 Solving, with RAG tool + memory)
Verifier Agent created (with RAG tool + System_Prompt3 + memory)

PHASE 1: Planning — CoT Step Parsing
Sending planning request to LLM (direct call, no agent)...
Phase 1 raw response length: 3597 chars
Identified 4 CoT step(s)
  Saved: outputs\CoT_step_query1.txt
  Saved: outputs\CoT_ste